# Build a Baseline Regression Helper

This is the regression twin of lesson 3.1. Same pattern: the LLM is an assistant, not the trainer. It looks at a short profile of your data and suggests **three regressors worth trying**, each with minimal constructor kwargs. 

## 1 - Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -q google-genai pandas scikit-learn python-dotenv


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../02-Preprocessing-Helpers").resolve()))

In [ ]:
import importlib
import json
import os
import re
import warnings

import pandas as pd
from dotenv import load_dotenv
from google import genai
from preprocessing_pipeline import preprocessing_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold, cross_val_score

warnings.filterwarnings("ignore")

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

GEN_CONFIG = {"temperature": 0.0, "seed": 42}

In [ ]:
result = preprocessing_pipeline(
    raw_path="../../data/hr_analytics.csv",
    target_col="MonthlyIncome",
    task="regression",
    column_types={"Attrition": "binary"},
)
X_train = result.X_train_enc
X_test = result.X_test_enc
y_train = result.y_train
y_test = result.y_test


## 2 - Profile the Data

Before asking the LLM anything, we build a short profile of the training data: size, feature mix, target statistics (mean/std/min/max), correlation, skew, and two quick probe scores — one from a linear model, one from a shallow tree. The gap between the two probes hints at how non-linear the problem is.

In [ ]:
def profile_data(X_train, y_train):
    """Build a short profile the LLM can reason about."""
    import numpy as np

    X_df = pd.DataFrame(X_train)
    y = pd.Series(y_train)

    n_numeric = int(X_df.select_dtypes(include="number").shape[1])
    n_categorical = int(X_df.shape[1] - n_numeric)

    numeric = X_df.select_dtypes(include="number")
    skew_max = float(numeric.skew().abs().max()) if numeric.shape[1] else 0.0
    low_var = int((numeric.var() < 0.01).sum()) if numeric.shape[1] else 0

    n_to_p_ratio = round(X_df.shape[0] / max(X_df.shape[1], 1), 2)

    corr = X_df.corr().abs().to_numpy(copy=True)
    np.fill_diagonal(corr, np.nan)
    max_corr = float(np.nanmax(corr))

    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    linear_probe = LinearRegression()
    tree_probe = RandomForestRegressor(
        n_estimators=100, max_depth=6, random_state=42, n_jobs=-1
    )
    linear_r2 = float(
        cross_val_score(linear_probe, X_train, y_train, cv=cv, scoring="r2").mean()
    )
    tree_r2 = float(
        cross_val_score(tree_probe, X_train, y_train, cv=cv, scoring="r2").mean()
    )

    return {
        "rows": int(X_df.shape[0]),
        "features": int(X_df.shape[1]),
        "n_numeric": n_numeric,
        "n_categorical": n_categorical,
        "n_to_p_ratio": n_to_p_ratio,
        "target_mean": round(float(y.mean()), 3),
        "target_std": round(float(y.std()), 3),
        "target_min": round(float(y.min()), 3),
        "target_max": round(float(y.max()), 3),
        "target_skew": round(float(y.skew()), 3),
        "max_feature_correlation": round(max_corr, 3),
        "skewness_max": round(skew_max, 3),
        "low_variance_features": low_var,
       : round(tree_r2, 3), "linear_probe_r2": round(linear_r2, 3),
        "tree_probe_r2"
    }

In [ ]:
profile = profile_data(X_train, y_train)
profile

In [ ]:
SELECTION_PROMPT = (
    "You are a machine learning engineer. "
    "Based ONLY on the dataset profile below (not general ML advice), suggest EXACTLY 3 "
    "regressors to try as baselines. You may choose ANY scikit-learn-compatible regressor "
    "(sklearn.*, xgboost.XGBRegressor, etc). Pick the three you think best fit THIS profile.\n\n"
    "For each candidate return:\n"
    "  - module: full import path, e.g. 'sklearn.ensemble'\n"
    "  - class:  class name,        e.g. 'RandomForestRegressor'\n"
    "  - reason: MUST cite specific field names and values from the profile. "
    "If the reason could apply to any dataset, rewrite it.\n"
    "  - init_kwargs: a dict of constructor kwargs to use as-is (e.g. random_state=42). "
    "Keep this minimal — we are training baselines with defaults, not tuning.\n\n"
    'Return ONLY valid JSON: {"candidates": [{"module": str, "class": str, "reason": str, '
    '"init_kwargs": {...}}, ...]}'
)

## 3 - Ask the LLM for 3 Candidates

We give the LLM the profile and ask it to suggest **three regressors** that fit this dataset. For each one it returns `module` + `class` (full import path), a profile-grounded `reason`, and minimal `init_kwargs`. No fixed allowlist — the LLM can pick anything sklearn-compatible. Python validates by importing the class.

In [ ]:
def pick_candidates(profile):
    """Ask the LLM for 3 regressors. Validate by actually importing the class."""
    content = f"Profile: {json.dumps(profile)}"
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=content,
        config={"system_instruction": SELECTION_PROMPT, **GEN_CONFIG},
    )
    text = re.sub(r"```(?:json)?", "", response.text or "").replace("```", "").strip()
    picks = json.loads(text)["candidates"]
    assert len(picks) == 3, f"Expected 3 candidates, got {len(picks)}"
    for p in picks:
        # Validate the class is real by importing it — guardrail against hallucination.
        mod = importlib.import_module(p["module"])
        p["cls"] = getattr(mod, p["class"])
        p.setdefault("init_kwargs", {})
        # Pin random_state so scores are reproducible regardless of what the LLM returned.
        # LinearRegression has no random_state; ignore kwarg errors at fit-time by only setting it
        # when the class signature accepts it.
        import inspect as _inspect

        if "random_state" in _inspect.signature(p["cls"]).parameters:
            p["init_kwargs"]["random_state"] = 42
    return picks

In [ ]:
candidates = pick_candidates(profile)
for c in candidates:
    print(f"- {c['module']}.{c['class']}  init_kwargs={c['init_kwargs']}")
    print(f"    reason: {c['reason']}")

## 4 - Score Each Candidate with Cross-Validation

For each of the 3 regressors, we fit with the LLM's suggested `init_kwargs` and score with 5-fold CV on RMSE (lower is better). No hyperparameter tuning here — that comes in 3.3.

In [ ]:
def score_candidate(candidate, X_train, y_train):
    """Fit + 5-fold CV on RMSE. No tuning."""
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    reg = candidate["cls"](**candidate["init_kwargs"])
    rmse = -cross_val_score(
        reg, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
    ).mean()
    return {
        "model": f"{candidate['module']}.{candidate['class']}",
        "cls": candidate["cls"],
        "init_kwargs": candidate["init_kwargs"],
        "rmse": rmse,
    }

In [ ]:
leaderboard = (
    pd.DataFrame([score_candidate(c, X_train, y_train) for c in candidates])
    .sort_values("rmse")
    .reset_index(drop=True)
)

leaderboard[["model", "rmse", "init_kwargs"]]

## 5 - Fit the Winner

The top row of the leaderboard is our baseline. We refit it on the full training set. We report one headline number (RMSE) here — deep evaluation lives in Module 4.

In [ ]:
winner = leaderboard.iloc[0]
model = winner["cls"](**winner["init_kwargs"])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
rmse = root_mean_squared_error(y_test, y_pred)
print(f"{winner['model']} — test RMSE: {rmse:.3f}")


## 6 - Wrap It All in One Function

We just walked through the steps one by one: profile the data, ask the LLM for 3 candidates, score each with CV, pick the winner, fit, and score. Now let's wrap that whole flow into a single function `regression_helper` so we can reuse it in later lessons with one call.

In [ ]:
def regression_helper(X_train, X_test, y_train, y_test):
    """Run the full flow: profile → LLM picks 3 → CV-score → fit winner → return results."""
    profile = profile_data(X_train, y_train)
    candidates = pick_candidates(profile)

    leaderboard = (
        pd.DataFrame([score_candidate(c, X_train, y_train) for c in candidates])
        .sort_values("rmse")
        .reset_index(drop=True)
    )

    winner = leaderboard.iloc[0]
    model = winner["cls"](**winner["init_kwargs"])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        "model": model,
        "model_name": winner["model"],
        "init_kwargs": winner["init_kwargs"],
        "profile": profile,
        "candidates": candidates,
        "leaderboard": leaderboard,
        "y_pred": y_pred,
    }

In [ ]:
result = regression_helper(X_train, X_test, y_train, y_test)
print("Winner:", result["model_name"])
print("Test RMSE:", round(root_mean_squared_error(y_test, result["y_pred"]), 3))

## 7 - Save for Reuse

Dump the helper (plus its imports and the functions it depends on) into `regression_helper.py` so later lessons can `from regression_helper import regression_helper`.

In [ ]:
import inspect

components = [
    "import importlib",
    "import json",
    "import os",
    "import re",
    "",
    "import numpy as np",
    "import pandas as pd",
    "from dotenv import load_dotenv",
    "from google import genai",
    "from sklearn.ensemble import RandomForestRegressor",
    "from sklearn.linear_model import LinearRegression",
    "from sklearn.metrics import root_mean_squared_error",
    "from sklearn.model_selection import KFold, cross_val_score",
    "",
    "load_dotenv()",
    "client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))",
    "",
    f"GEN_CONFIG = {GEN_CONFIG!r}",
    "",
    f"SELECTION_PROMPT = {SELECTION_PROMPT!r}",
    "",
    inspect.getsource(profile_data),
    "",
    inspect.getsource(pick_candidates),
    "",
    inspect.getsource(score_candidate),
    "",
    inspect.getsource(regression_helper),
]

with open("regression_helper.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved regression_helper.py")